In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
from os.path import join
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
from datetime import datetime, timedelta

def days_in_week_range(week_number, year):
    """
    Helper to count days in a week. If it's the current week, return days up to today.
    week_number: int week number (ISO, 1-53).
    year: int year.
    current_week: int, current week number.
    """

    # Get Monday of the ISO week
    try:
        start_date = date.fromisocalendar(int(year), int(week_number), 1)
    except Exception:
        # If not a valid ISO week/year, fall back to 7
        return 7

    # End on Sunday
    end_date = start_date + timedelta(days=6)

    today = datetime.now().date()
    this_iso = today.isocalendar()[:2]  # (year, week)

    if (int(year), int(week_number)) == this_iso:
        delta = (today - start_date).days + 1  # include today
        return min(max(delta, 0), 7)
    else:
        return 7

In [ ]:
customer_name = 'Cadent'
customer_id = Query(query = f"SELECT CustomerId FROM KPI_Customer WHERE Name = '{customer_name}'").execute([KPIHub_Conn]).values[0][0]

customer_utilization = {'Hours':7, 'Days':7}
tableList = [KPI_ReportSummary,KPI_SurveySummary]
aggregator = ['BoundaryRegion','ReportYear', 'ReportWeek']
data = {}
for table in tableList:
    data[table] = Query(query = f"SELECT * FROM {table.name} WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId IN (SELECT CustomerId FROM KPI_Customer WHERE Name = '{customer_name}'))").execute([KPIHub_Conn])

def report_KPI_apply(df):
    if hasattr(df, 'name') and df.name is not None:
        group_year = df.name[0]
        group_week_range = df.name[1]
    else:
        group_year, group_week_range = None, None
    current_year = datetime.now().year

    current_week = datetime.now().isocalendar()[1]
    if group_year is not None and group_week_range is not None:
        day_count = days_in_week_range(group_week_range, current_year)
    else:
        day_count = None
    
    unique_reports = df.drop_duplicates(subset=['ReportId'])
    no_surveyors = df['SurveyorUnit'].nunique()
    surveyDurationHours = df['SurveyDurationMinutes'].sum()/60
    targetTimeHours = day_count*customer_utilization['Hours']*no_surveyors
    starndardTargetTimeHours = 6*5*25
    #targetTimeHours = 7*7*25
    surveyCount = df['SurveyId'].nunique()
    avg_speed_weighted = df['AvgSpeedKm'] * df['TotalSegments']
    print(df.name, no_surveyors, surveyCount)
    # Simplified: precompute all potentially zero divisors, and ensure safe division in all places
    days = day_count if day_count else 0
    surveyors = no_surveyors if no_surveyors else 0
    total_segments = df['TotalSegments'].sum()
    total_km = df['TotalKilometers'].sum()
    survey_min = df['SurveyDurationMinutes'].sum()
    asset_covered = unique_reports['AssetCoveredLengthKm'].sum()
    dist_pipe_covered = unique_reports['DistributionPipeCoveredKm'].sum()
    night_km = df['NightKilometers'].sum()
    day_km = df['DayKilometers'].sum()

    return pd.Series({
        'SurveyDurationHours': surveyDurationHours,
        'TargetDurationHours': targetTimeHours,
        'CustomerUtilization': surveyDurationHours/targetTimeHours if targetTimeHours else 0,
        'StarndardUtilization': surveyDurationHours/starndardTargetTimeHours if starndardTargetTimeHours else 0,
        'TotalSurveyors': surveyors,
        'ProductivityPerSurveyor': dist_pipe_covered/surveyors if surveyors else 0,
        'SurveyCount': surveyCount,
        'AvgSpeedKm': avg_speed_weighted.sum()/total_segments if total_segments else 0,
        'SurveysCarDay': surveyCount/surveyors/days if surveyors and days else 0,
        'IdleTime': 100*df['IdleTimeMinutes'].sum()/survey_min if survey_min else 0,
        'DaysCount': days,
        'TotalDrivenLengthKm': total_km,
        'DrivingRatio': total_km/asset_covered if asset_covered else 0,
        'NightDrivenLength': night_km,
        'DayDrivenLength': day_km,
        'NightRatio': 100*night_km/total_km if total_km else 0,
        'DayRatio': 100*day_km/total_km if total_km else 0,
    })
report_summary_agg = pd.merge(data[KPI_ReportSummary],data[KPI_SurveySummary],on='ReportId',how="left")
report_KPI_df = report_summary_agg.groupby(aggregator).apply(report_KPI_apply)
report_KPI_df = report_KPI_df.drop(columns=["DaysCount"]).round(2)

('East Midlands', 2024, 16) 1 15
('East Midlands', 2024, 35) 1 7
('East Midlands', 2024, 36) 1 10
('East Midlands', 2024, 48) 1 10
('East Midlands', 2026, 5) 2 16
('East Midlands', 2026, 6) 2 48
('East Midlands', 2026, 7) 2 53
('East Midlands', 2026, 8) 2 63
('East Midlands', 2026, 9) 2 51
('East Midlands', 2026, 10) 2 50
('East Midlands', 2026, 11) 3 36
('East Midlands', 2026, 12) 5 56
('East Midlands', 2026, 13) 5 135
('East Midlands', 2026, 14) 5 81
('East Midlands', 2026, 15) 6 101
('East Midlands', 2026, 16) 6 190
('East Midlands', 2026, 17) 6 152
('East Midlands', 2026, 18) 5 149
('East Midlands', 2026, 19) 5 106
('East Midlands', 2026, 20) 5 118
('East Midlands', 2026, 21) 5 117
('East Midlands', 2026, 22) 5 106
('East Midlands', 2026, 23) 5 130
('East Midlands', 2026, 24) 6 127
('East Midlands', 2026, 25) 6 136
('East Midlands', 2026, 26) 6 177
('East Midlands', 2026, 27) 6 143
('East Midlands', 2026, 28) 3 16
('East of England', 2026, 5) 1 9
('East of England', 2026, 6) 1 19
(

In [5]:
report_to_check = report_summary_agg[(report_summary_agg['ReportYear']==2025) & (report_summary_agg['ReportWeek']==43)]
report_to_check.head()

,ReportId,CustomerId,ReportName,ReportDate,ReportYear,ReportMonth,ReportWeek,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,...,IdleSegments,TotalSegments,TotalKilometers,DayKilometers,NightKilometers,SegmentDurationMinutes,IdleTimeMinutes,ActiveTimeMinutes,AvgSpeedKm,LastUpdated_y
6348,67113563-8F11-DE4D-E9E3-3A1D21D7E509,BD4D080B-1D12-D329-ABD0-39FEB9804E98,CR-671135,2025-10-23 06:04:19.593000,2025,10,43,49.226880,47.833365,48.273375,...,6,310,58.687697,9.347319,49.340378,159.203067,11.770850,147.432217,29.979037,2026-07-02 12:27:07.382177
6349,67113563-8F11-DE4D-E9E3-3A1D21D7E509,BD4D080B-1D12-D329-ABD0-39FEB9804E98,CR-671135,2025-10-23 06:04:19.593000,2025,10,43,49.226880,47.833365,48.273375,...,5,300,57.053520,0.000000,57.053520,149.695400,12.927383,136.768017,29.173491,2026-07-02 12:27:07.382177
6350,67113563-8F11-DE4D-E9E3-3A1D21D7E509,BD4D080B-1D12-D329-ABD0-39FEB9804E98,CR-671135,2025-10-23 06:04:19.593000,2025,10,43,49.226880,47.833365,48.273375,...,12,301,55.824215,0.000000,55.824215,171.218900,37.676567,133.542333,27.411872,2026-07-02 12:27:07.382177
10512,AD67E8A7-2E90-8AAA-27A6-3A1D21DBD38E,BD4D080B-1D12-D329-ABD0-39FEB9804E98,CR-AD67E8,2025-10-23 06:08:37.263000,2025,10,43,43.633908,39.408141,41.546415,...,1,50,9.813470,0.000000,9.813470,36.535333,19.807000,16.728333,36.791443,2026-07-02 12:27:07.382177
10513,AD67E8A7-2E90-8AAA-27A6-3A1D21DBD38E,BD4D080B-1D12-D329-ABD0-39FEB9804E98,CR-AD67E8,2025-10-23 06:08:37.263000,2025,10,43,43.633908,39.408141,41.546415,...,8,287,55.071899,8.410374,46.661524,127.977750,27.761783,100.215967,38.111489,2026-07-02 12:27:07.382177


In [6]:
r = report_KPI_df.reset_index()
r_long = r.melt(id_vars=aggregator, var_name='KPIId', value_name='Value')
r_long['Id'] = r_long.apply(lambda row: f"{row['KPIId']}_{customer_name}R{row['BoundaryRegion'].replace(' ','')}_Y{row['ReportYear']}_W{row['ReportWeek']}", axis=1)
r_long = r_long.rename(columns={'ReportWeek': 'PeriodValue', 'ReportYear': 'Year'})
r_long['PeriodType'] = "Weekly"
r_long['LastUpdated'] = datetime.now()
r_long['CustomerId'] = customer_id


In [7]:
r_long

,BoundaryRegion,Year,PeriodValue,KPIId,Value,Id,PeriodType,LastUpdated,CustomerId
0,East Midlands,2024,16,SurveyDurationHours,45.54,SurveyDurationHours_CadentREastMidlands_Y2024_W16,Weekly,2026-07-06 14:27:21.814534,BD4D080B-1D12-D329-ABD0-39FEB9804E98
1,East Midlands,2024,35,SurveyDurationHours,24.57,SurveyDurationHours_CadentREastMidlands_Y2024_W35,Weekly,2026-07-06 14:27:21.814534,BD4D080B-1D12-D329-ABD0-39FEB9804E98
2,East Midlands,2024,36,SurveyDurationHours,34.58,SurveyDurationHours_CadentREastMidlands_Y2024_W36,Weekly,2026-07-06 14:27:21.814534,BD4D080B-1D12-D329-ABD0-39FEB9804E98
3,East Midlands,2024,48,SurveyDurationHours,20.79,SurveyDurationHours_CadentREastMidlands_Y2024_W48,Weekly,2026-07-06 14:27:21.814534,BD4D080B-1D12-D329-ABD0-39FEB9804E98
4,East Midlands,2026,5,SurveyDurationHours,51.11,SurveyDurationHours_CadentREastMidlands_Y2026_W5,Weekly,2026-07-06 14:27:21.814534,BD4D080B-1D12-D329-ABD0-39FEB9804E98
...,...,...,...,...,...,...,...,...,...
4507,West Midlands,2026,24,DayRatio,0.00,DayRatio_CadentRWestMidlands_Y2026_W24,Weekly,2026-07-06 14:27:21.814534,BD4D080B-1D12-D329-ABD0-39FEB9804E98
4508,West Midlands,2026,25,DayRatio,0.00,DayRatio_CadentRWestMidlands_Y2026_W25,Weekly,2026-07-06 14:27:21.814534,BD4D080B-1D12-D329-ABD0-39FEB9804E98
4509,West Midlands,2026,26,DayRatio,0.00,DayRatio_CadentRWestMidlands_Y2026_W26,Weekly,2026-07-06 14:27:21.814534,BD4D080B-1D12-D329-ABD0-39FEB9804E98
4510,West Midlands,2026,27,DayRatio,0.00,DayRatio_CadentRWestMidlands_Y2026_W27,Weekly,2026-07-06 14:27:21.814534,BD4D080B-1D12-D329-ABD0-39FEB9804E98


In [8]:
KPI_Data.update_table(arguments = {'DataFrame': r_long, 'db_path': DB_PATH, 'PrimaryKey': 'Id'})
KPI_Data.query_table(arguments = {'db_path': DB_PATH})


,Id,KPIId,CustomerId,BoundaryRegion,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,FOVMain_Cadent_Y2023_W14,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Weekly,14,None,None,2026-07-06 14:26:22.674023
1,FOVMain_Cadent_Y2023_W15,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Weekly,15,None,None,2026-07-06 14:26:22.674023
2,FOVMain_Cadent_Y2023_W16,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Weekly,16,None,None,2026-07-06 14:26:22.674023
3,FOVMain_Cadent_Y2023_W17,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Weekly,17,None,None,2026-07-06 14:26:22.674023
4,FOVMain_Cadent_Y2023_W19,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,None,2023,Weekly,19,None,None,2026-07-06 14:26:22.674023
...,...,...,...,...,...,...,...,...,...,...
24160,DayRatio_CadentRWestMidlands_Y2026_W24,DayRatio,BD4D080B-1D12-D329-ABD0-39FEB9804E98,West Midlands,2026,Weekly,24,0.0,None,2026-07-06 14:27:21.814534
24161,DayRatio_CadentRWestMidlands_Y2026_W25,DayRatio,BD4D080B-1D12-D329-ABD0-39FEB9804E98,West Midlands,2026,Weekly,25,0.0,None,2026-07-06 14:27:21.814534
24162,DayRatio_CadentRWestMidlands_Y2026_W26,DayRatio,BD4D080B-1D12-D329-ABD0-39FEB9804E98,West Midlands,2026,Weekly,26,0.0,None,2026-07-06 14:27:21.814534
24163,DayRatio_CadentRWestMidlands_Y2026_W27,DayRatio,BD4D080B-1D12-D329-ABD0-39FEB9804E98,West Midlands,2026,Weekly,27,0.0,None,2026-07-06 14:27:21.814534
